# Conversation history, memory editing, Mem0, and RLM (Chapter 8)

This notebook covers the chapter's four memory strategies: explicit `dspy.History`, summarization of old turns, selective retention of ReAct results, long-term memory with Mem0, and programmatic context exploration with `dspy.RLM`.

**Required environment variable:** `OPENAI_API_KEY`.


In [ ]:
%pip install -r ../requirements.txt -q


In [ ]:
%pip install mem0ai -q


In [ ]:
import dspy
from dotenv import load_dotenv

load_dotenv()

dspy.configure(lm=dspy.LM("openai/gpt-5.6-luna"))


## Conversation history with `dspy.History`


In [ ]:
class ChatBot(dspy.Signature):
    """Answer the user while maintaining context from earlier turns."""

    question: str = dspy.InputField()
    history: dspy.History = dspy.InputField()
    answer: str = dspy.OutputField()


chatbot = dspy.Predict(ChatBot)

response1 = chatbot(
    question="What is the capital of France?",
    history=dspy.History(messages=[]),
)

history = dspy.History(messages=[{
    "question": "What is the capital of France?",
    "answer": response1.answer,
}])

response2 = chatbot(
    question="What's the population of that city?",
    history=history,
)
print(response2.answer)


## Memory editing

Keep recent turns verbatim and compress older turns into a factual summary. Each history dictionary uses the same `question` and `answer` keys as the `ChatBot` signature.


In [ ]:
class SummarizeOldTurns(dspy.Signature):
    """Summarize older turns without inventing information.

    Preserve current goals, constraints, unresolved issues, and corrections.
    When details conflict, keep the newest version.
    """

    previous_summary: str = dspy.InputField()
    turns: list[dict[str, str]] = dspy.InputField()
    summary: str = dspy.OutputField()


class HistoryEditor:
    def __init__(self, keep_last_n_turns: int = 3):
        self.keep_last_n_turns = max(1, keep_last_n_turns)
        self.turns: list[dict[str, str]] = []
        self.summary = ""
        self.summarize = dspy.Predict(SummarizeOldTurns)

    def add_turn(self, question: str, answer: str) -> None:
        self.turns.append({
            "question": question,
            "answer": answer,
        })

        if len(self.turns) > self.keep_last_n_turns:
            expired = self.turns[:-self.keep_last_n_turns]
            self.summary = self.summarize(
                previous_summary=self.summary,
                turns=expired,
            ).summary
            self.turns = self.turns[-self.keep_last_n_turns:]

    def build(self) -> dspy.History:
        messages = self.turns.copy()

        if self.summary:
            messages.insert(0, {
                "question": "What should you remember from earlier?",
                "answer": self.summary,
            })

        return dspy.History(messages=messages)


In [ ]:
editor = HistoryEditor(keep_last_n_turns=3)

question = "What programming language should I use?"
history_result = chatbot(question=question, history=editor.build())
editor.add_turn(question, history_result.answer)
print(history_result.answer)


## Editing tool results

Keep the full ReAct trajectory in logs, but put only the final user-facing turn into conversational history. This prevents large tool payloads from being replayed on every later call.


In [ ]:
def web_search(query: str) -> str:
    """Search the web for current information."""
    return f"[stub] Current DSPy release information for: {query}"


react = dspy.ReAct(
    "question -> answer",
    tools=[web_search],
    max_iters=6,
)

question = "What changed in the latest DSPy release?"
react_result = react(question=question)

# Store the useful result, not react_result.trajectory.
editor.add_turn(
    question=question,
    answer=react_result.answer,
)


## Long-term memory with Mem0


In [ ]:
try:
    from mem0 import Memory

    config = {
        "llm": {
            "provider": "openai",
            "config": {"model": "gpt-5.6-luna", "temperature": 0.1},
        },
        "embedder": {
            "provider": "openai",
            "config": {"model": "text-embedding-3-small"},
        },
    }
    memory = Memory.from_config(config)
    mem0_ready = True
except Exception as exc:
    memory = None
    mem0_ready = False
    print(f"Could not initialize Mem0: {exc!r}")


In [ ]:
def store_memory(content: str, user_id: str = "default") -> str:
    """Store important information in long-term memory for later recall."""
    memory.add(content, user_id=user_id)
    return f"Stored: {content}"


def search_memories(query: str, user_id: str = "default") -> str:
    """Search long-term memory for information relevant to the query."""
    response = memory.search(
        query,
        filters={"user_id": user_id},
        top_k=5,
    )
    results = response.get("results", [])

    if not results:
        return "No relevant memories found."

    return "\n".join(result["memory"] for result in results)


def get_all_memories(user_id: str = "default") -> str:
    """Retrieve all stored memories for a user."""
    response = memory.get_all(filters={"user_id": user_id})
    results = response.get("results", [])

    if not results:
        return "No memories stored."

    return "\n".join(result["memory"] for result in results)


In [ ]:
class MemoryAgent(dspy.Signature):
    """You are a helpful assistant with long-term memory.
    Store important user information and preferences.
    Search your memory before answering questions about the user.
    """

    user_input: str = dspy.InputField()
    response: str = dspy.OutputField()


if mem0_ready:
    memory_agent = dspy.ReAct(
        MemoryAgent,
        tools=[store_memory, search_memories, get_all_memories],
        max_iters=6,
    )

    memory_agent(
        user_input="My name is Mike and I prefer Python over JavaScript."
    )
    memory_result = memory_agent(
        user_input="What programming language should you write examples in for me?"
    )
    print(memory_result.response)
else:
    print("Skipping the Mem0 agent because initialization failed.")


## Context management with `dspy.RLM`


In [ ]:
very_long_text = (
    "Q1 customer retention report... "
    "Across enterprise accounts churn dropped 12% quarter over quarter, "
    "driven primarily by the new onboarding revamp. Mid-market churn was flat. "
    "Self-serve churn increased 4%, concentrated in trials that never connected "
    "a data source."
)

rlm = dspy.RLM(
    "documents, question -> answer",
    max_iterations=10,
    max_llm_calls=50,
    sub_lm=dspy.LM("openai/gpt-5.6-luna"),
)

rlm_result = rlm(
    documents=very_long_text,
    question="What were the key findings about customer retention?",
)
print(rlm_result.answer)
